# 🏠 House Price Prediction
**Project by: Charan G**  
B.Tech CSE | NMAM Institute of Technology, Nitte

---
### Models Used:
1. Linear Regression
2. Ridge Regression
3. Decision Tree Regressor
4. Random Forest Regressor
5. Gradient Boosting Regressor
6. XGBoost Regressor

### Dataset: King County Style House Price Dataset (2000 records, 19 features)


## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install xgboost if not already installed
# !pip install xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# XGBoost
from xgboost import XGBRegressor

print("✅ All libraries imported successfully!")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")

## 📊 Step 2: Generate / Load Dataset

In [ ]:
# -------------------------------------------------------
# OPTION A: Load from CSV (if you downloaded the dataset)
# df = pd.read_csv('house_price_dataset.csv')
# -------------------------------------------------------

# OPTION B: Generate dataset inline (no file needed)
np.random.seed(42)
n = 2000

bedrooms      = np.random.choice([1,2,3,4,5,6], n, p=[0.05,0.20,0.35,0.25,0.10,0.05])
bathrooms     = np.clip(bedrooms - np.random.randint(0,2,n) + np.random.randint(0,2,n), 1, 5).astype(float)
sqft_living   = np.round(bedrooms * 400 + np.random.normal(500, 300, n)).clip(500, 8000)
sqft_lot      = np.round(np.random.lognormal(8.5, 0.8, n)).clip(1000, 100000)
floors        = np.random.choice([1,1.5,2,2.5,3], n, p=[0.35,0.10,0.35,0.10,0.10])
waterfront    = np.random.choice([0,1], n, p=[0.95,0.05])
view          = np.random.choice([0,1,2,3,4], n, p=[0.65,0.10,0.10,0.10,0.05])
condition     = np.random.choice([1,2,3,4,5], n, p=[0.02,0.08,0.50,0.30,0.10])
grade         = np.random.choice(range(3,13), n, p=[0.01,0.02,0.10,0.20,0.30,0.20,0.10,0.04,0.02,0.01])
sqft_above    = np.round(sqft_living * np.random.uniform(0.6, 1.0, n)).clip(500, sqft_living)
sqft_basement = (sqft_living - sqft_above).clip(0)
yr_built      = np.random.randint(1900, 2023, n)
yr_renovated  = np.where(np.random.rand(n) < 0.1, np.random.randint(1970,2023,n), 0)
zipcode       = np.random.choice(range(98001, 98200), n)
lat           = np.random.uniform(47.1, 47.8, n)
long_         = np.random.uniform(-122.5, -121.3, n)
sqft_living15 = np.round(sqft_living * np.random.uniform(0.8, 1.2, n)).clip(500, 6000)
sqft_lot15    = np.round(sqft_lot * np.random.uniform(0.7, 1.3, n)).clip(500, 100000)

price = (
    sqft_living * 150
    + bedrooms * 8000
    + bathrooms * 12000
    + floors * 15000
    + waterfront * 180000
    + view * 20000
    + condition * 10000
    + grade * 25000
    + (2023 - yr_built) * (-300)
    + (lat - 47.1) * 120000
    + np.random.normal(0, 40000, n)
).clip(80000, 2500000)
price = np.round(price, -3)

df = pd.DataFrame({
    'bedrooms':       bedrooms,
    'bathrooms':      bathrooms,
    'sqft_living':    sqft_living.astype(int),
    'sqft_lot':       sqft_lot.astype(int),
    'floors':         floors,
    'waterfront':     waterfront,
    'view':           view,
    'condition':      condition,
    'grade':          grade,
    'sqft_above':     sqft_above.astype(int),
    'sqft_basement':  sqft_basement.astype(int),
    'yr_built':       yr_built,
    'yr_renovated':   yr_renovated,
    'zipcode':        zipcode,
    'lat':            np.round(lat, 4),
    'long':           np.round(long_, 4),
    'sqft_living15':  sqft_living15.astype(int),
    'sqft_lot15':     sqft_lot15.astype(int),
    'price':          price.astype(int)
})

print(f"✅ Dataset created: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Shape      : {df.shape}")
print(f"Missing    : {df.isnull().sum().sum()}")
print(f"Duplicates : {df.duplicated().sum()}")
print()
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribution of Key Features', fontsize=16, fontweight='bold')

features = ['price', 'sqft_living', 'bedrooms', 'bathrooms', 'grade', 'yr_built']
colors   = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4']

for ax, feat, color in zip(axes.flatten(), features, colors):
    ax.hist(df[feat], bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=12)
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots – Price vs Top Features
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Price vs Key Features', fontsize=14, fontweight='bold')

scatter_feats = [('sqft_living', '#2196F3'), ('grade', '#4CAF50'), ('lat', '#E91E63')]

for ax, (feat, color) in zip(axes, scatter_feats):
    ax.scatter(df[feat], df['price'], alpha=0.3, color=color, s=15)
    ax.set_xlabel(feat.replace('_', ' ').title())
    ax.set_ylabel('Price ($)')
    ax.set_title(f'Price vs {feat.replace("_", " ").title()}')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with price
print("\nTop Correlations with Price:")
print(corr['price'].abs().sort_values(ascending=False).drop('price').head(8))

## 🛠️ Step 4: Feature Engineering

In [ ]:
df_model = df.copy()

# Derived features
df_model['house_age']         = 2024 - df_model['yr_built']
df_model['years_since_reno']  = np.where(df_model['yr_renovated'] > 0,
                                          2024 - df_model['yr_renovated'], 
                                          df_model['house_age'])
df_model['was_renovated']     = (df_model['yr_renovated'] > 0).astype(int)
df_model['has_basement']      = (df_model['sqft_basement'] > 0).astype(int)
df_model['rooms_total']       = df_model['bedrooms'] + df_model['bathrooms']
df_model['sqft_per_room']     = df_model['sqft_living'] / (df_model['rooms_total'] + 1)
df_model['lot_to_living']     = df_model['sqft_lot'] / df_model['sqft_living']

# Drop raw columns replaced by engineered ones
df_model.drop(columns=['yr_built', 'yr_renovated', 'zipcode'], inplace=True)

print("✅ Feature engineering done!")
print(f"Features: {df_model.shape[1] - 1} input columns + 1 target (price)")
print(df_model.columns.tolist())

## ✂️ Step 5: Train / Test Split & Scaling

In [ ]:
X = df_model.drop(columns=['price'])
y = df_model['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train size : {X_train.shape}")
print(f"Test size  : {X_test.shape}")

## 🤖 Step 6: Train 6 Models

In [ ]:
# ============================================================
#  Model 1 – Linear Regression
# ============================================================
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
pred_lr = lr.predict(X_test_sc)
print("Model 1: Linear Regression")
print(f"  R²  = {r2_score(y_test, pred_lr):.4f}")
print(f"  MAE = ${mean_absolute_error(y_test, pred_lr):,.0f}")
print(f"  RMSE= ${np.sqrt(mean_squared_error(y_test, pred_lr)):,.0f}")

In [ ]:
# ============================================================
#  Model 2 – Ridge Regression (L2 Regularization)
# ============================================================
ridge = Ridge(alpha=10.0)
ridge.fit(X_train_sc, y_train)
pred_ridge = ridge.predict(X_test_sc)
print("Model 2: Ridge Regression")
print(f"  R²  = {r2_score(y_test, pred_ridge):.4f}")
print(f"  MAE = ${mean_absolute_error(y_test, pred_ridge):,.0f}")
print(f"  RMSE= ${np.sqrt(mean_squared_error(y_test, pred_ridge)):,.0f}")

In [ ]:
# ============================================================
#  Model 3 – Decision Tree Regressor
# ============================================================
dt = DecisionTreeRegressor(max_depth=8, min_samples_split=10, random_state=42)
dt.fit(X_train, y_train)   # trees don't need scaling
pred_dt = dt.predict(X_test)
print("Model 3: Decision Tree Regressor")
print(f"  R²  = {r2_score(y_test, pred_dt):.4f}")
print(f"  MAE = ${mean_absolute_error(y_test, pred_dt):,.0f}")
print(f"  RMSE= ${np.sqrt(mean_squared_error(y_test, pred_dt)):,.0f}")

In [ ]:
# ============================================================
#  Model 4 – Random Forest Regressor
# ============================================================
rf = RandomForestRegressor(n_estimators=200, max_depth=12,
                           min_samples_split=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print("Model 4: Random Forest Regressor")
print(f"  R²  = {r2_score(y_test, pred_rf):.4f}")
print(f"  MAE = ${mean_absolute_error(y_test, pred_rf):,.0f}")
print(f"  RMSE= ${np.sqrt(mean_squared_error(y_test, pred_rf)):,.0f}")

In [ ]:
# ============================================================
#  Model 5 – Gradient Boosting Regressor
# ============================================================
gbr = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                 max_depth=5, subsample=0.8, random_state=42)
gbr.fit(X_train, y_train)
pred_gbr = gbr.predict(X_test)
print("Model 5: Gradient Boosting Regressor")
print(f"  R²  = {r2_score(y_test, pred_gbr):.4f}")
print(f"  MAE = ${mean_absolute_error(y_test, pred_gbr):,.0f}")
print(f"  RMSE= ${np.sqrt(mean_squared_error(y_test, pred_gbr)):,.0f}")

In [ ]:
# ============================================================
#  Model 6 – XGBoost Regressor
# ============================================================
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                    reg_lambda=1.0, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)
print("Model 6: XGBoost Regressor")
print(f"  R²  = {r2_score(y_test, pred_xgb):.4f}")
print(f"  MAE = ${mean_absolute_error(y_test, pred_xgb):,.0f}")
print(f"  RMSE= ${np.sqrt(mean_squared_error(y_test, pred_xgb)):,.0f}")

## 📈 Step 7: Model Comparison

In [ ]:
def evaluate(name, y_true, y_pred):
    return {
        'Model':  name,
        'R²':     round(r2_score(y_true, y_pred), 4),
        'MAE':    round(mean_absolute_error(y_true, y_pred), 0),
        'RMSE':   round(np.sqrt(mean_squared_error(y_true, y_pred)), 0),
    }

results = pd.DataFrame([
    evaluate('Linear Regression',         y_test, pred_lr),
    evaluate('Ridge Regression',          y_test, pred_ridge),
    evaluate('Decision Tree',             y_test, pred_dt),
    evaluate('Random Forest',             y_test, pred_rf),
    evaluate('Gradient Boosting',         y_test, pred_gbr),
    evaluate('XGBoost',                   y_test, pred_xgb),
])

results = results.sort_values('R²', ascending=False).reset_index(drop=True)
results['Rank'] = results.index + 1
print("\n" + "=" * 60)
print("           MODEL COMPARISON SUMMARY")
print("=" * 60)
print(results.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Comparison', fontsize=15, fontweight='bold')

colors = ['#EF5350', '#AB47BC', '#42A5F5', '#26A69A', '#FF7043', '#66BB6A']

# R²
axes[0].barh(results['Model'], results['R²'], color=colors)
axes[0].set_xlabel('R² Score (higher is better)')
axes[0].set_title('R² Score')
axes[0].set_xlim(0, 1)
for i, v in enumerate(results['R²']):
    axes[0].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)

# MAE
axes[1].barh(results['Model'], results['MAE'] / 1000, color=colors)
axes[1].set_xlabel('MAE ($ thousands) — lower is better')
axes[1].set_title('Mean Absolute Error')
for i, v in enumerate(results['MAE']):
    axes[1].text(v/1000 + 0.2, i, f'${v/1000:.1f}k', va='center', fontsize=9)

# RMSE
axes[2].barh(results['Model'], results['RMSE'] / 1000, color=colors)
axes[2].set_xlabel('RMSE ($ thousands) — lower is better')
axes[2].set_title('Root Mean Squared Error')
for i, v in enumerate(results['RMSE']):
    axes[2].text(v/1000 + 0.2, i, f'${v/1000:.1f}k', va='center', fontsize=9)

for ax in axes:
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 🔮 Step 8: Predicted vs Actual (Best Model)

In [ ]:
# Use the best model from results
best_name = results.iloc[0]['Model']
model_map = {
    'Linear Regression': pred_lr,
    'Ridge Regression':  pred_ridge,
    'Decision Tree':     pred_dt,
    'Random Forest':     pred_rf,
    'Gradient Boosting': pred_gbr,
    'XGBoost':           pred_xgb,
}
best_pred = model_map[best_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'Best Model: {best_name}', fontsize=14, fontweight='bold')

# Predicted vs Actual
axes[0].scatter(y_test, best_pred, alpha=0.4, color='#2196F3', s=20)
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect Fit')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Residuals
residuals = y_test - best_pred
axes[1].hist(residuals, bins=40, color='#FF9800', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Residual (Actual − Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✅ Best Model: {best_name}")
print(f"   R²  = {results.iloc[0]['R²']}")
print(f"   MAE = ${results.iloc[0]['MAE']:,.0f}")
print(f"   RMSE= ${results.iloc[0]['RMSE']:,.0f}")

## 📊 Step 9: Feature Importance (Random Forest)

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=True)

plt.figure(figsize=(10, 8))
colors_imp = ['#EF9A9A' if v < feat_imp.median() else '#42A5F5' for v in feat_imp]
feat_imp.plot(kind='barh', color=colors_imp, edgecolor='white')
plt.title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
print(feat_imp.sort_values(ascending=False).head())

## 🔁 Step 10: Cross-Validation (5-Fold)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Linear Regression':  (LinearRegression(),          X_train_sc),
    'Ridge Regression':   (Ridge(alpha=10.0),            X_train_sc),
    'Decision Tree':      (DecisionTreeRegressor(max_depth=8, random_state=42), X_train),
    'Random Forest':      (RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), X_train),
    'Gradient Boosting':  (GradientBoostingRegressor(n_estimators=100, random_state=42), X_train),
    'XGBoost':            (XGBRegressor(n_estimators=100, random_state=42, verbosity=0), X_train),
}

cv_results = {}
for name, (model, X_data) in cv_models.items():
    scores = cross_val_score(model, X_data, y_train, cv=kf, scoring='r2')
    cv_results[name] = scores
    print(f"{name:25s}  CV R²: {scores.mean():.4f} ± {scores.std():.4f}")

print("\n✅ Cross-validation complete!")

In [ ]:
# Box plot of CV scores
plt.figure(figsize=(12, 6))
cv_df = pd.DataFrame(cv_results)
cv_df.boxplot(vert=False, patch_artist=True,
              boxprops=dict(facecolor='#90CAF9'),
              medianprops=dict(color='red', lw=2))
plt.xlabel('R² Score')
plt.title('5-Fold Cross-Validation R² Distribution', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

## 🏠 Step 11: Predict a Custom House

In [ ]:
# ---- Define your house here ----
custom_house = {
    'bedrooms':       3,
    'bathrooms':      2.0,
    'sqft_living':    1800,
    'sqft_lot':       5000,
    'floors':         1.0,
    'waterfront':     0,
    'view':           1,
    'condition':      3,
    'grade':          7,
    'sqft_above':     1800,
    'sqft_basement':  0,
    'lat':            47.5,
    'long':          -122.0,
    'sqft_living15':  1700,
    'sqft_lot15':     4800,
    # Engineered features
    'house_age':      20,
    'years_since_reno': 20,
    'was_renovated':  0,
    'has_basement':   0,
    'rooms_total':    5.0,
    'sqft_per_room':  1800 / 6,
    'lot_to_living':  5000 / 1800,
}

custom_df = pd.DataFrame([custom_house])
# Make sure column order matches
custom_df = custom_df[X.columns]

# Scale for linear models
custom_sc = scaler.transform(custom_df)

preds_custom = {
    'Linear Regression': lr.predict(custom_sc)[0],
    'Ridge Regression':  ridge.predict(custom_sc)[0],
    'Decision Tree':     dt.predict(custom_df)[0],
    'Random Forest':     rf.predict(custom_df)[0],
    'Gradient Boosting': gbr.predict(custom_df)[0],
    'XGBoost':           xgb.predict(custom_df)[0],
}

print("\n🏠 Custom House Prediction:")
print(f"   Bedrooms: {custom_house['bedrooms']} | Bathrooms: {custom_house['bathrooms']}")
print(f"   Sqft: {custom_house['sqft_living']} | Grade: {custom_house['grade']} | Age: {custom_house['house_age']} yrs")
print()
print(f"{'Model':<25} {'Predicted Price':>20}")
print("-" * 48)
for model_name, pred in preds_custom.items():
    print(f"{model_name:<25} ${pred:>18,.0f}")
print("-" * 48)
avg = np.mean(list(preds_custom.values()))
print(f"{'Ensemble Average':<25} ${avg:>18,.0f}")

## ✅ Step 12: Final Summary

In [ ]:
print("="*60)
print("       🏠 HOUSE PRICE PREDICTION — FINAL SUMMARY")
print("="*60)
print(f"  Dataset        : 2,000 houses × 19 features")
print(f"  Train/Test     : 80% / 20%")
print(f"  Target         : Sale Price ($)")
print()
print(results[['Rank','Model','R²','MAE','RMSE']].to_string(index=False))
print()
best = results.iloc[0]
print(f"  🏆 Best Model  : {best['Model']}")
print(f"     R²          = {best['R²']}  (explains {best['R²']*100:.1f}% of price variance)")
print(f"     MAE         = ${best['MAE']:,.0f}  (avg prediction error)")
print(f"     RMSE        = ${best['RMSE']:,.0f}")
print("="*60)
print("  Project by: Charan G | NMIT, Nitte | CSE B.Tech")
print("="*60)